# 第12回：改善実験を小さく回す

**今日の問い：改善した理由を後から説明できる実験とは何か。**

**セルの動かし方**：各セル（灰色の枠）を選んで `Shift + Enter`（またはセル左の▷ボタン）を押すと実行できます。
**上から順に**実行してください。前のセルを飛ばすと、後のセルでエラーになります。

`TRY`は全員、`CHANGE`は値を1つ変える練習、`CHALLENGE`は余裕がある人向けです。
`DEEP DIVE`・`APPENDIX`は経験者や自習向けの発展で、飛ばしても本編は完結します。
分からないコードは、セル全体ではなく気になる数行をM365 Copilotへ貼って相談します。


In [ ]:
# 【準備セル】教材フォルダの場所を自動で見つけます。中身は今は理解しなくてOK、そのまま実行してください。
from pathlib import Path

def find_repo_root(start=Path.cwd()):
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise FileNotFoundError("pyproject.tomlがある勉強会フォルダ内で実行してください")

ROOT = find_repo_root()
DATA = ROOT / "data"
print("教材フォルダ:", ROOT)


## この回でできるようになること

- 変更を1要素に限定した比較を設計し、実験ログを関数で残す
- RandomizedSearchCVで探索し、ネストCVで楽観の少ない推定を得る
- 並べ替え重要度の信頼区間とエラー分析から次の仮説を選ぶ

### 進み方

`CORE`は同期90分で扱う本線、`DEEP DIVE`は時間があれば扱う深掘り、
`SELF-STUDY`は任意自習です。すべて終わらなくても次回へ進めます。
経験者は`CORE`を早めに終え、`DEEP DIVE`を5人で分担して読むと深まります。

### 先に押さえる言葉

- 実験ログ：変更・条件・結果・解釈を残す記録
- ランダム探索：候補を無作為に試すハイパーパラメータ探索
- ネストCV：探索と評価を分けて過大評価を防ぐ交差検証
- 信頼区間：推定値の不確かさを表す幅
- 再現性：同じ手順で同じ結果を得られる性質

> **実行前の30秒予想**：今日の問いに、今の言葉で仮の答えを書いてから始めます。


## 「なんとなく良くなった」を卒業する

改善は勢いでやると、後で「なぜ良くなったのか」を説明できません。この回のテーマは、**理由を後から
説明できる実験のやり方**です。次の原則が効きます。

1. **一度に変えるのは1つだけ**（複数変えると、どれが効いたか分からない）。
2. **比較条件は固定**（同じ分割・同じ指標）。
3. **結果は平均とばらつきで残す**（1回のスコアで一喜一憂しない）。
4. **良くなった実験も悪くなった実験も記録する**（消さない）。

まず、実験の土台（データ・交差検証）を用意します。


In [ ]:
import pandas as pd

df = pd.read_csv(DATA / "compound_experiments.csv")
print(f"{len(df)}行 × {len(df.columns)}列")
df.head()


In [ ]:
from sklearn.model_selection import cross_validate, StratifiedKFold
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance

features = ["temperature_c", "reaction_time_h", "concentration_m", "molecular_weight", "logp", "tpsa"]
X, y = df[features], df["active"]
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)


## TRY：1要素だけ変えて、実験ログに残す

`max_depth`**だけ**を変えた3つの実験を回し、結果を表（実験ログ）にします。他の設定は固定。
学習F1と検証F1平均を両方残すのは、**過学習の度合い**（第6回）も一緒に記録するためです。


In [ ]:
rows = []
for depth in [3, 6, None]:
    model = make_pipeline(SimpleImputer(strategy="median"), RandomForestClassifier(n_estimators=150, max_depth=depth, random_state=42))
    scores = cross_validate(model, X, y, cv=cv, scoring="f1", return_train_score=True)
    rows.append({"実験名": f"depth={depth}", "変更点": "max_depthのみ",
                 "学習F1": scores["train_score"].mean(), "検証F1平均": scores["test_score"].mean(),
                 "検証F1標準偏差": scores["test_score"].std()})
experiment_log = pd.DataFrame(rows)
experiment_log.round(3)


### 出力の読み方

- **検証F1平均が最も高い深さ**が候補。ただし**検証F1標準偏差**が大きいなら、その優位は不安定かもしれません。
- 差が標準偏差より小さいなら「実質同じ」と読み、より単純な（浅い）設定を選ぶのが無難です。
- `depth=None`で学習F1が跳ね上がり検証F1が伸びないなら、過学習。**表1つで「効果」と「過学習」を同時に管理**できます。


## 実験ログの最小項目

同期回でも自習でも、次を1行で残せば十分です。

- **実験名 / 変えたもの（1つ）/ 固定した比較条件 / 結果の平均とばらつき / 気づき / 次の仮説**

Copilotには次の実験案を出してもらってもよいですが、**優先順位と「予測時点で妥当か」の判断は人**が行います。


## DEEP DIVE：探索を自動化し、正直な推定を得る

手で`max_depth`を変えるのは学習には良いですが、設定が増えると大変です。**探索の自動化**と、
第6回で学んだ**ネストCV（正直な推定）**、そして**重要度を区間で読む**ことを扱います。


### RandomizedSearchCV：設定を自動で探す

複数の設定候補から無作為に組み合わせを試し、交差検証で最良を選びます。総当たり（GridSearch）より
少ない回数で広く探せるのが利点。`n_iter`が試行回数です。


In [ ]:
from sklearn.model_selection import RandomizedSearchCV

pipe = make_pipeline(SimpleImputer(strategy="median"), RandomForestClassifier(random_state=42))
param_dist = {
    "randomforestclassifier__n_estimators": [100, 200, 300],
    "randomforestclassifier__max_depth": [3, 4, 6, None],
    "randomforestclassifier__min_samples_leaf": [1, 2, 4],
    "randomforestclassifier__max_features": ["sqrt", "log2", None],
}
search = RandomizedSearchCV(pipe, param_dist, n_iter=10, cv=cv, scoring="f1", random_state=42)
search.fit(X, y)
print("最良設定:", search.best_params_)
print("探索内での最良CV F1:", round(search.best_score_, 3))


### 出力の読み方と、大事な注意

`best_params_`が選ばれた設定、`best_score_`がそのCF1です。**ただしこの`best_score_`をそのまま「性能」として
報告してはいけません**。たくさん試して一番良かった数字なので、下駄を履いています。次で正直な推定に直します。


### ネストCV：探索の下駄を脱いだ推定

「探索」を1つのモデルとみなし、その外側でもう一段の交差検証をかけます。各外側分割で設定を選び直し、
未見のデータで評価するので、**探索による楽観が乗らない正直な性能**が得られます。


In [ ]:
from sklearn.model_selection import cross_val_score

outer = StratifiedKFold(5, shuffle=True, random_state=7)
nested = cross_val_score(search, X, y, cv=outer, scoring="f1")
print("ネストCV外側F1:", nested.round(3))
print("楽観の少ない推定:", round(nested.mean(), 3), "±", round(nested.std(), 3), " ← 探索内スコアより低いのが普通")


### 出力の読み方

ネストCVの平均は、前セルの`best_score_`より**少し低い**のが普通で、その差が「探索による楽観」の大きさです。
論文や報告に載せるなら、こちらの正直な数字を使います。


### 並べ替え重要度は「区間」で読む

第1回で見た並べ替え重要度を、今度は**ばらつき（±2SD）つき**で読みます。下限が0を跨ぐ特徴量は
「効いているとは言い切れない」。評価は学習に使っていない**holdout**で行い、公平性を保ちます。


In [ ]:
from sklearn.model_selection import train_test_split

X_fit, X_holdout, y_fit, y_holdout = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
best = make_pipeline(SimpleImputer(strategy="median"), RandomForestClassifier(n_estimators=200, max_depth=6, random_state=42)).fit(X_fit, y_fit)
perm = permutation_importance(best, X_holdout, y_holdout, scoring="f1", n_repeats=30, random_state=42)
importance = pd.DataFrame({
    "特徴量": features,
    "重要度平均": perm.importances_mean,
    "下限(平均-2SD)": perm.importances_mean - 2 * perm.importances_std,
}).sort_values("重要度平均", ascending=False)
importance["0を跨ぐ"] = importance["下限(平均-2SD)"] <= 0
importance.round(4)


### 出力の読み方

- `0を跨ぐ=True`の特徴量は、**寄与があるとは断言できない**（ばらつきの範囲に0が入る）。
- 上位で`0を跨ぐ=False`の特徴量が、自信を持って「効いている」と言える列。ここから**反証可能な次の仮説**
（「この列を強める特徴量を足したら改善するのでは？」）を1つ立てて、COREの実験ログへ戻ります。これが改善サイクルです。


## APPENDIX（任意・追加演習）

実験の回し方を仕組み化します。90分の外の自習向けです。まず**実験を1行で記録する関数**を作り、
複数の設定を回してログに溜めます。手作業のコピペより、記録漏れが減ります。


In [ ]:
from sklearn.model_selection import cross_val_score

experiment_log = []
def run_experiment(name, estimator, note=""):
    "設定を交差検証で評価し、実験ログへ1行追加して返す。"
    scores = cross_val_score(estimator, X, y, cv=cv, scoring="f1")
    row = {"実験名": name, "F1平均": round(scores.mean(), 3), "F1_SD": round(scores.std(), 3), "気づき": note}
    experiment_log.append(row)
    return row

run_experiment("depth3", make_pipeline(SimpleImputer(strategy="median"), RandomForestClassifier(n_estimators=150, max_depth=3, random_state=42)), "浅め")
run_experiment("depth6", make_pipeline(SimpleImputer(strategy="median"), RandomForestClassifier(n_estimators=150, max_depth=6, random_state=42)), "標準")
run_experiment("leaf4", make_pipeline(SimpleImputer(strategy="median"), RandomForestClassifier(n_estimators=150, max_depth=6, min_samples_leaf=4, random_state=42)), "葉を大きく")
pd.DataFrame(experiment_log)


### 出力の読み方

3つの実験がログにたまり、F1平均・ばらつき・気づきが1表に。**変更点と結果がセットで残る**ので、後から
「なぜこの設定にしたか」を説明できます。関数化しておくと、実験のたびに1行呼ぶだけで済みます。


### 検証曲線：1つの設定を動かして最適点を探す

`validation_curve`は、1つのハイパーパラメータ（ここでは`max_depth`）を動かし、学習と検証のスコア推移を
描きます。最適な複雑さが視覚的に分かります。


In [ ]:
import matplotlib.pyplot as plt
from sklearn.model_selection import validation_curve

depths = [2, 3, 4, 6, 8, 12]
tr, va = validation_curve(
    make_pipeline(SimpleImputer(strategy="median"), RandomForestClassifier(n_estimators=150, random_state=42)),
    X, y, param_name="randomforestclassifier__max_depth", param_range=depths, cv=cv, scoring="f1",
)
plt.plot(depths, tr.mean(1), "o-", label="学習")
plt.plot(depths, va.mean(1), "o-", label="検証")
plt.xlabel("max_depth"); plt.ylabel("F1"); plt.legend(); plt.title("検証曲線")
plt.tight_layout()


### 出力の読み方

学習F1は深さとともに上がり続けますが、検証F1は途中で頭打ち・下降します。**検証F1が最大になる手前**が
最適な深さ。2本の乖離が広がるほど過学習が進んでいる、という第6回の読み方がそのまま使えます。


### 実験ログをファイルに残す

ログをCSVに保存し、読み直します。セッションをまたいで実験を積み上げられ、再現性（第15回）にもつながります。


In [ ]:
out = ROOT / "workspace" / "experiment_log.csv"
pd.DataFrame(experiment_log).to_csv(out, index=False)
reloaded = pd.read_csv(out)
print("保存＆再読込した実験ログ:", out)
display(reloaded)


### 出力の読み方

`workspace/experiment_log.csv`に保存され、読み直しても同じ内容。**記録を残す文化**が、思いつきの改善を
再現可能な知見へ変えます。良い変更も悪い変更も、まずログに残すことを、この回でいちばんの習慣にしてください。


## よくある誤り

- 同時に複数要素を変える
- 探索に使った分割で最終性能も報告する
- 悪化した実験を記録から消す

## SELF-STUDY（任意・30〜60分）

- RandomizedSearchCVの最良設定を、ネストCVの外側スコアで確かめる
- 並べ替え重要度を20反復で計算し、区間が0を跨ぐ列を挙げる

成果は完成したコードでなくても、予想・変更点・出力・解釈を4行で残せば十分です。

## 振り返りチェック

1. 1要素だけ変える理由は何か
2. ネストCVは何を防ぐか
3. 重要度の区間が0を跨ぐとどう解釈するか

答えに詰まった項目が、次に見返す場所です。暗記ではなくNotebookの該当セルを指せればOKです。
